# RecoverWave — Notebook 01: Whoop Personal Data Characterisation**Author:** Harry Diss**Project:** COM6001 Final Year Project (Pathway A)**Research Question:** Do the audio characteristics of music listened to in the 2-hour window following a workout predict next-day Whoop recovery score?---## Purpose of This NotebookBefore building any ML model, we must characterise the personal dataset we're working with. This notebook:1. **Loads** all 4 Whoop CSV exports (`physiological_cycles.csv`, `workouts.csv`, `sleeps.csv`, `journal_entries.csv`)2. **Validates** data integrity and identifies gaps3. **Profiles** four physiological domains:   - Recovery score (the modelling target)   - HRV (the primary autonomic nervous system marker)   - Sleep architecture (deep / REM / light / awake)   - Strain and workout patterns4. **Establishes baselines** for the n=1 sample size discussion in the dissertation5. **Produces publication-quality figures** for inclusion in the Literature Survey and Design sections of the report## Why This Matters (Dissertation Context)The COM6001 marking criteria weight *Reliability & Validity* at 20%. A rigorous characterisation of the personal dataset directly supports this criterion by:- Documenting sample size, temporal coverage, and missingness transparently- Comparing personal baselines to published population distributions (e.g. HRV norms from [Nunan et al. (2010)](https://pubmed.ncbi.nlm.nih.gov/20718611/))- Identifying potential confounds (illness, travel, alcohol) before model training- Providing the empirical basis for the *Limitations* section> **Expected runtime:** 1–3 minutes depending on export size (typically <1MB for 12 months of data).

## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Make src importable (adjust if running from a different location)
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT))

# Data & plotting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import timedelta

# RecoverWave modules
from src.whoop_parser import WhoopParser

# Plot config — consistent styling for dissertation figures
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "axes.titleweight": "bold",
    "figure.titleweight": "bold",
    "legend.frameon": True,
    "legend.framealpha": 0.9,
})
sns.set_palette("husl")

# Directory for saving figures for the dissertation
FIG_DIR = PROJECT_ROOT / "data" / "processed" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Directory where your Whoop CSVs live
WHOOP_DIR = PROJECT_ROOT / "data" / "whoop"

print(f"Project root: {PROJECT_ROOT}")
print(f"Whoop data:   {WHOOP_DIR}")
print(f"Figure dir:   {FIG_DIR}")
print(f"Files found:  {sorted(p.name for p in WHOOP_DIR.glob('*.csv'))}")

## 2. Load All 4 CSVsWhoop exports (as of 2026) contain:| File | One row per | Key fields ||---|---|---|| `physiological_cycles.csv` | Physiological cycle (sleep-to-sleep) | recovery score, HRV, RHR, daily strain, kilojoules || `workouts.csv` | Individual workout | start/end time, strain, avg/max HR, sport, kJ, HR zones || `sleeps.csv` | Sleep (main + naps) | start/end, light/SWS/REM durations, efficiency, disturbances || `journal_entries.csv` | Journal response | question ID, answer (often boolean: caffeine, alcohol, stress) |Whoop organises data by **physiological cycles** (sleep-to-sleep), not calendar days. Our parser extracts a `date` column from the cycle start time to enable day-level alignment with Spotify listening history.

In [ ]:
parser = WhoopParser(str(WHOOP_DIR))
data = parser.load_all()

cycles   = data["cycles"]
workouts = data["workouts"]
sleeps   = data["sleeps"]
journal  = data["journal"]

print(f"\nShapes:")
for name, df in data.items():
    print(f"  {name:10s}: {df.shape[0]:>5} rows × {df.shape[1]:>3} cols")

In [ ]:
# Peek at column names so we know what's available in your specific export
print("=== CYCLES ===")
print(list(cycles.columns))
print("\n=== WORKOUTS ===")
print(list(workouts.columns))
print("\n=== SLEEPS ===")
print(list(sleeps.columns))
print("\n=== JOURNAL ===")
print(list(journal.columns) if len(journal) else "(empty)")

## 3. Temporal Coverage & Data QualityBefore any analysis, we establish:- **Total days** in the export (coverage)- **Missing days** (gaps where the strap wasn't worn or failed to sync)- **Quality flags** per metricThis is critical for the *Reliability & Validity* section — n=1 studies must be transparent about data density.

In [ ]:
# Build a continuous daily timeline
timeline = parser.build_daily_timeline()

# Temporal bounds
date_min = pd.to_datetime(timeline["date"]).min()
date_max = pd.to_datetime(timeline["date"]).max()
span_days = (date_max - date_min).days + 1

# Missingness per key metric
key_metrics = [c for c in [
    "recovery_score",
    "hrv_rmssd_milliseconds",
    "resting_heart_rate",
    "day_strain",
    "sleep_performance_percentage",
] if c in timeline.columns]

missingness = timeline[key_metrics].isnull().sum()
coverage_pct = (1 - missingness / len(timeline)) * 100

print(f"Export spans: {date_min.date()} → {date_max.date()} ({span_days} days)")
print(f"Rows in timeline: {len(timeline)}")
print(f"\nCoverage by metric:")
for col, miss, cov in zip(key_metrics, missingness, coverage_pct):
    bar = "█" * int(cov / 5) + "░" * (20 - int(cov / 5))
    print(f"  {col:38s} {bar} {cov:5.1f}%  ({miss} missing)")

In [ ]:
# Visualise coverage as a calendar heatmap (one row per week)
if "recovery_score" in timeline.columns:
    timeline["date_dt"] = pd.to_datetime(timeline["date"])
    timeline["week"]    = timeline["date_dt"].dt.isocalendar().week
    timeline["year"]    = timeline["date_dt"].dt.isocalendar().year
    timeline["weekday"] = timeline["date_dt"].dt.weekday  # 0=Mon

    pivot = timeline.pivot_table(
        index=["year", "week"],
        columns="weekday",
        values="recovery_score",
        aggfunc="first",
    )

    fig, ax = plt.subplots(figsize=(11, max(4, len(pivot) * 0.18)))
    sns.heatmap(
        pivot,
        cmap="RdYlGn",
        vmin=0, vmax=100,
        cbar_kws={"label": "Recovery Score (%)"},
        linewidths=0.4, linecolor="white",
        xticklabels=["Mon","Tue","Wed","Thu","Fri","Sat","Sun"],
        yticklabels=False,
        ax=ax,
    )
    ax.set_title("Recovery Score — Calendar Heatmap (missing days appear white/grey)")
    ax.set_xlabel("Day of Week")
    ax.set_ylabel("Week (chronological)")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig_01_coverage_heatmap.png", dpi=300, bbox_inches="tight")
    plt.show()

## 4. Recovery Score — The Modelling TargetRecovery score (0–100%) is Whoop's composite daily readiness metric, weighted from HRV, RHR, respiratory rate, and sleep performance. It is the **target variable** for RecoverWave.**Official Whoop zones:**- **Green** (67–100%): body ready for high strain- **Yellow** (34–66%): moderate strain recommended- **Red** (0–33%): recovery/rest recommendedFor a regression model, we care about:- **Central tendency** (is your distribution well-behaved?)- **Spread** (enough variance to learn from?)- **Autocorrelation** (does today predict tomorrow? — sets the baseline to beat)- **Day-of-week effects** (does recovery dip after weekend activities?)

In [ ]:
if "recovery_score" in timeline.columns:
    rec = timeline["recovery_score"].dropna()

    stats = {
        "n":           len(rec),
        "mean":        rec.mean(),
        "median":      rec.median(),
        "std":         rec.std(),
        "min":         rec.min(),
        "max":         rec.max(),
        "% green ≥67": (rec >= 67).mean() * 100,
        "% yellow":    ((rec >= 34) & (rec < 67)).mean() * 100,
        "% red <34":   (rec < 34).mean() * 100,
    }

    print("Recovery Score — Summary Statistics")
    print("-" * 42)
    for k, v in stats.items():
        print(f"  {k:14s}: {v:7.2f}")

    # Compare to population norms (Whoop reports population median ~55%)
    print("\nPopulation context:")
    print("  Whoop reports typical member median ≈ 55%")
    print(f"  Your median is {stats['median']:.0f}% → "
          f"{'above' if stats['median'] > 55 else 'below'} population median")

In [ ]:
# 4-panel recovery profile figure
if "recovery_score" in timeline.columns:
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))

    # (a) Distribution with zones
    ax = axes[0, 0]
    ax.hist(rec, bins=25, edgecolor="white", alpha=0.85, color="#4C72B0")
    ax.axvspan(67, 100, alpha=0.12, color="green", label="Green ≥67%")
    ax.axvspan(34, 67,  alpha=0.12, color="gold",  label="Yellow 34–66%")
    ax.axvspan(0,  34,  alpha=0.12, color="red",   label="Red <34%")
    ax.axvline(rec.mean(),   color="navy", linestyle="--", linewidth=1.5, label=f"Mean {rec.mean():.0f}%")
    ax.axvline(rec.median(), color="black", linestyle=":", linewidth=1.5, label=f"Median {rec.median():.0f}%")
    ax.set_title("(a) Recovery Score Distribution")
    ax.set_xlabel("Recovery Score (%)"); ax.set_ylabel("Frequency")
    ax.legend(fontsize=8, loc="upper left")

    # (b) Time series
    ax = axes[0, 1]
    ts = timeline.dropna(subset=["recovery_score"]).sort_values("date_dt")
    ax.plot(ts["date_dt"], ts["recovery_score"], alpha=0.6, linewidth=0.9, color="#4C72B0")
    ax.plot(ts["date_dt"], ts["recovery_score"].rolling(7, min_periods=3).mean(),
            linewidth=2.2, color="#DD5555", label="7-day rolling mean")
    ax.axhline(67, color="green", linestyle="--", alpha=0.5)
    ax.axhline(34, color="red",   linestyle="--", alpha=0.5)
    ax.set_title("(b) Recovery Over Time")
    ax.set_xlabel("Date"); ax.set_ylabel("Recovery (%)")
    ax.legend()
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

    # (c) Autocorrelation (lag-1 to lag-14)
    ax = axes[1, 0]
    from pandas.plotting import autocorrelation_plot
    series = ts["recovery_score"].reset_index(drop=True)
    lags = range(1, 15)
    acf = [series.autocorr(lag=l) for l in lags]
    ax.bar(list(lags), acf, color="#55A868", edgecolor="black", linewidth=0.5)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.axhline(1.96/np.sqrt(len(series)),  color="red", linestyle="--", alpha=0.6,
               label="95% CI")
    ax.axhline(-1.96/np.sqrt(len(series)), color="red", linestyle="--", alpha=0.6)
    ax.set_title("(c) Autocorrelation of Recovery Score")
    ax.set_xlabel("Lag (days)"); ax.set_ylabel("Autocorrelation")
    ax.legend()

    # (d) Day-of-week pattern
    ax = axes[1, 1]
    ts["dow"] = ts["date_dt"].dt.day_name()
    day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
    sns.boxplot(data=ts, x="dow", y="recovery_score", order=day_order, ax=ax,
                palette="viridis")
    ax.set_title("(d) Recovery by Day of Week")
    ax.set_xlabel(""); ax.set_ylabel("Recovery (%)")
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

    plt.suptitle("Personal Recovery Score Profile", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig_02_recovery_profile.png", dpi=300, bbox_inches="tight")
    plt.show()

    # Persistence baseline: predicting tomorrow = today
    lag1 = series.autocorr(lag=1)
    print(f"\nLag-1 autocorrelation: {lag1:.3f}")
    print(f"Persistence baseline (predict tomorrow = today) MAE: "
          f"{(series.diff().abs().mean()):.2f}")
    print("→ Our ML model MUST beat this MAE to be worth keeping.")

## 5. HRV (RMSSD) — The Autonomic Marker**HRV RMSSD** (root mean square of successive differences between normal R-R intervals, measured in milliseconds) is the single most important input to Whoop's recovery calculation and the primary physiological marker linking music to the autonomic nervous system.### Literature Context- Young healthy adults (20–30): typical RMSSD **42–65 ms** ([Nunan et al., 2010, *Pacing Clin Electrophysiol*](https://pubmed.ncbi.nlm.nih.gov/20718611/))- Elite endurance athletes: often **>80 ms**- Music research: slow/melodic music increases HRV RMSSD in listeners ([Frontiers in Cardiovascular Medicine, 2021](https://pmc.ncbi.nlm.nih.gov/articles/PMC8417899/))The HRV distribution in your personal data sets the *baseline autonomic state* that post-workout music may modulate. Wide variance is helpful — it gives the model signal to learn from.

In [ ]:
hrv_col = "hrv_rmssd_milliseconds" if "hrv_rmssd_milliseconds" in timeline.columns else None

# Whoop sometimes labels this slightly differently in exports
if hrv_col is None:
    candidates = [c for c in timeline.columns if "hrv" in c.lower() or "rmssd" in c.lower()]
    if candidates:
        hrv_col = candidates[0]
        print(f"Using HRV column: {hrv_col}")

if hrv_col:
    hrv = timeline[hrv_col].dropna()

    print("HRV RMSSD — Summary Statistics")
    print("-" * 42)
    print(f"  n:      {len(hrv)}")
    print(f"  mean:   {hrv.mean():.1f} ms")
    print(f"  median: {hrv.median():.1f} ms")
    print(f"  std:    {hrv.std():.1f} ms")
    print(f"  IQR:    {hrv.quantile(0.25):.1f} – {hrv.quantile(0.75):.1f} ms")
    print(f"  range:  {hrv.min():.1f} – {hrv.max():.1f} ms")

    # Compare to population norms
    pop_median_young = 53  # approx from Nunan et al. 2010, ages 20-30
    print(f"\nPopulation reference (Nunan et al. 2010, ages 20-30): median ≈ {pop_median_young} ms")
    print(f"Your median: {hrv.median():.0f} ms → "
          f"{hrv.median() / pop_median_young * 100:.0f}% of population median")
else:
    print("⚠  No HRV column found — skipping HRV analysis.")

In [ ]:
if hrv_col:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    # (a) Distribution with population comparison
    ax = axes[0]
    ax.hist(hrv, bins=25, edgecolor="white", alpha=0.85, color="#8172B2")
    ax.axvline(hrv.median(), color="navy", linestyle="--", linewidth=1.5,
               label=f"Your median: {hrv.median():.0f} ms")
    ax.axvspan(42, 65, alpha=0.15, color="green",
               label="Healthy adult range (Nunan 2010)")
    ax.set_title("(a) HRV RMSSD Distribution")
    ax.set_xlabel("RMSSD (ms)"); ax.set_ylabel("Frequency")
    ax.legend(fontsize=8)

    # (b) HRV time series with 7-day rolling mean
    ax = axes[1]
    ts_hrv = timeline.dropna(subset=[hrv_col]).sort_values("date_dt")
    ax.plot(ts_hrv["date_dt"], ts_hrv[hrv_col], alpha=0.5, linewidth=0.8, color="#8172B2")
    ax.plot(ts_hrv["date_dt"], ts_hrv[hrv_col].rolling(7, min_periods=3).mean(),
            linewidth=2.2, color="#CC4444", label="7-day rolling mean")
    ax.set_title("(b) HRV Over Time")
    ax.set_xlabel("Date"); ax.set_ylabel("RMSSD (ms)")
    ax.legend()
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

    # (c) HRV vs Recovery score relationship
    ax = axes[2]
    both = timeline.dropna(subset=[hrv_col, "recovery_score"])
    ax.scatter(both[hrv_col], both["recovery_score"],
               alpha=0.5, s=18, color="#8172B2", edgecolor="white", linewidth=0.3)
    # Best fit line
    if len(both) > 2:
        z = np.polyfit(both[hrv_col], both["recovery_score"], 1)
        xs = np.linspace(both[hrv_col].min(), both[hrv_col].max(), 50)
        ax.plot(xs, np.polyval(z, xs), color="#CC4444", linewidth=2,
                label=f"r = {both[hrv_col].corr(both['recovery_score']):.2f}")
    ax.set_title("(c) HRV ↔ Recovery Score")
    ax.set_xlabel("RMSSD (ms)"); ax.set_ylabel("Recovery (%)")
    ax.legend()

    plt.suptitle("HRV Profile & Recovery Relationship", fontsize=13, y=1.03)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig_03_hrv_profile.png", dpi=300, bbox_inches="tight")
    plt.show()

## 6. Sleep ArchitectureSleep stage composition is a crucial input because **post-workout music may influence sleep onset and therefore subsequent REM/SWS ratios**, which in turn drive HRV and recovery the next morning. This creates a potential causal chain we'll want to probe later:> `post-workout music → sleep architecture → next-day HRV → recovery score`**Healthy adult norms** ([Hirshkowitz et al., 2015](https://pubmed.ncbi.nlm.nih.gov/26551974/); [Ohayon et al., 2017](https://pubmed.ncbi.nlm.nih.gov/29073412/)):| Stage | Typical share of total sleep ||---|---|| Light (N1 + N2) | 45–55% || Deep / SWS (N3) | 13–23% || REM | 20–25% || Awake in bed | <5% |Total sleep time: 7–9 hours recommended for adults 18–25.

In [ ]:
# Identify sleep stage columns (names vary slightly by export version)
sleep_cols = {
    "light":   next((c for c in sleeps.columns if "light_sleep_duration" in c), None),
    "deep":    next((c for c in sleeps.columns if "slow_wave_sleep_duration" in c or "deep_sleep_duration" in c), None),
    "rem":     next((c for c in sleeps.columns if "rem_sleep_duration" in c), None),
    "awake":   next((c for c in sleeps.columns if "awake_duration" in c), None),
    "total":   next((c for c in sleeps.columns if "in_bed_duration" in c or "total_sleep_duration" in c), None),
    "eff":     next((c for c in sleeps.columns if "sleep_efficiency_percentage" in c), None),
    "perf":    next((c for c in sleeps.columns if "sleep_performance_percentage" in c), None),
    "dist":    next((c for c in sleeps.columns if "disturbances" in c), None),
}

print("Detected sleep columns:")
for k, v in sleep_cols.items():
    print(f"  {k:6s} → {v}")

In [ ]:
# Filter to main sleeps only (exclude naps if nap column exists)
sleeps_main = sleeps.copy()
nap_col = next((c for c in sleeps.columns if c == "nap" or "is_nap" in c), None)
if nap_col:
    sleeps_main = sleeps_main[sleeps_main[nap_col] != True]
    print(f"Excluded {(sleeps[nap_col] == True).sum()} naps. Main sleeps: {len(sleeps_main)}")

# Summary of sleep metrics
summary_rows = []
for stage in ["light", "deep", "rem", "awake"]:
    c = sleep_cols[stage]
    if c and c in sleeps_main.columns:
        v = sleeps_main[c].dropna()
        if len(v) > 0:
            summary_rows.append({
                "Stage": stage.capitalize(),
                "Mean (min)":  f"{v.mean():.0f}",
                "Median (min)":f"{v.median():.0f}",
                "Std (min)":   f"{v.std():.0f}",
                "Min (min)":   f"{v.min():.0f}",
                "Max (min)":   f"{v.max():.0f}",
            })

pd.DataFrame(summary_rows)

In [ ]:
# Total sleep & efficiency summary
if sleep_cols["total"]:
    total = sleeps_main[sleep_cols["total"]].dropna() / 60  # → hours
    print(f"Total sleep time:   mean {total.mean():.2f} h  (median {total.median():.2f} h)")
    print(f"  Nights <7h: {(total < 7).sum()} ({(total < 7).mean()*100:.0f}%)")
    print(f"  Nights 7-9h: {((total >= 7) & (total <= 9)).sum()} ({((total >= 7) & (total <= 9)).mean()*100:.0f}%)")
    print(f"  Nights >9h: {(total > 9).sum()} ({(total > 9).mean()*100:.0f}%)")

if sleep_cols["eff"]:
    eff = sleeps_main[sleep_cols["eff"]].dropna()
    print(f"\nSleep efficiency:   mean {eff.mean():.1f}%  (median {eff.median():.1f}%)")

if sleep_cols["dist"]:
    dist = sleeps_main[sleep_cols["dist"]].dropna()
    print(f"Disturbances/night: mean {dist.mean():.1f}  (max {dist.max():.0f})")

In [ ]:
# Sleep architecture figure — 4 panels
if all(sleep_cols[k] for k in ["light","deep","rem","awake"]):
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))

    # (a) Stage composition as % — stacked bar (mean across all nights)
    ax = axes[0, 0]
    means = {stage: sleeps_main[sleep_cols[stage]].mean() for stage in ["light","deep","rem","awake"]}
    total_mean = sum(means.values())
    percents = {k: v / total_mean * 100 for k, v in means.items()}

    colors = {"light": "#9BB7D4", "deep": "#2E4A6B", "rem": "#C4A7E7", "awake": "#D97677"}
    bottom = 0
    for stage in ["deep","rem","light","awake"]:
        ax.barh("Your avg night", percents[stage], left=bottom,
                color=colors[stage], edgecolor="white",
                label=f"{stage.capitalize()} ({percents[stage]:.0f}%)")
        bottom += percents[stage]
    ax.set_xlim(0, 100)
    ax.set_xlabel("% of total sleep time")
    ax.set_title("(a) Sleep Stage Composition")
    ax.legend(loc="lower right", fontsize=9)

    # (b) Total sleep duration distribution
    ax = axes[0, 1]
    if sleep_cols["total"]:
        total_h = sleeps_main[sleep_cols["total"]].dropna() / 60
        ax.hist(total_h, bins=25, edgecolor="white", alpha=0.85, color="#5B8DBE")
        ax.axvspan(7, 9, alpha=0.18, color="green", label="Recommended 7-9h")
        ax.axvline(total_h.median(), color="navy", linestyle="--",
                   label=f"Median: {total_h.median():.1f}h")
        ax.set_title("(b) Total Sleep Duration")
        ax.set_xlabel("Hours"); ax.set_ylabel("Nights")
        ax.legend()

    # (c) Stage durations boxplot
    ax = axes[1, 0]
    stage_data = []
    stage_labels = []
    for stage in ["deep", "rem", "light", "awake"]:
        vals = sleeps_main[sleep_cols[stage]].dropna()
        if len(vals):
            stage_data.append(vals)
            stage_labels.append(stage.capitalize())
    bp = ax.boxplot(stage_data, labels=stage_labels, patch_artist=True, showfliers=False)
    for patch, stage in zip(bp["boxes"], ["deep","rem","light","awake"]):
        patch.set_facecolor(colors[stage])
        patch.set_alpha(0.7)
    ax.set_ylabel("Minutes")
    ax.set_title("(c) Sleep Stage Duration Distributions")

    # (d) Sleep performance vs recovery
    ax = axes[1, 1]
    if sleep_cols["perf"] and "recovery_score" in timeline.columns:
        # Merge sleep perf into timeline by date (this requires date column in sleeps)
        sleeps_main["_date"] = pd.to_datetime(
            sleeps_main[sleeps_main.columns[0]]).dt.date
        perf_daily = sleeps_main.groupby("_date")[sleep_cols["perf"]].first().reset_index()
        perf_daily.columns = ["date", "sleep_perf"]
        merged = timeline.merge(perf_daily, on="date", how="inner").dropna(
            subset=["sleep_perf","recovery_score"])
        if len(merged) > 5:
            ax.scatter(merged["sleep_perf"], merged["recovery_score"],
                       alpha=0.5, s=18, color="#5B8DBE", edgecolor="white", linewidth=0.3)
            r = merged["sleep_perf"].corr(merged["recovery_score"])
            z = np.polyfit(merged["sleep_perf"], merged["recovery_score"], 1)
            xs = np.linspace(merged["sleep_perf"].min(), merged["sleep_perf"].max(), 50)
            ax.plot(xs, np.polyval(z, xs), color="#CC4444", linewidth=2,
                    label=f"r = {r:.2f}")
            ax.set_title("(d) Sleep Performance ↔ Recovery")
            ax.set_xlabel("Sleep Performance (%)"); ax.set_ylabel("Recovery (%)")
            ax.legend()

    plt.suptitle("Sleep Architecture Profile", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig_04_sleep_architecture.png", dpi=300, bbox_inches="tight")
    plt.show()

## 7. Strain & Workout PatternsWhoop's **strain score (0–21)** is a cardiovascular load measure on an exponential curve:- 0–9   = Light (minimal cardiovascular demand)- 10–13 = Moderate- 14–17 = High- 18–21 = All-OutFor RecoverWave, workouts matter because:1. **Workout end time** defines the start of our 2-hour post-workout listening window2. **Workout intensity** (strain, max HR) is a confounder for recovery we must control for3. **Sport type** (gym vs hockey vs cardio) may interact with music effects differentlyUnderstanding the *workout density* in your dataset tells us how many aligned rows we'll have for modelling.

In [ ]:
print(f"Total workouts: {len(workouts)}")

if "workout_end_dt" in workouts.columns:
    first = workouts["workout_end_dt"].min()
    last  = workouts["workout_end_dt"].max()
    days  = (last - first).days + 1
    print(f"Workout span: {first.date()} → {last.date()} ({days} days)")
    print(f"Workouts per week: {len(workouts) / days * 7:.1f}")

# Sport distribution
sport_col = next((c for c in workouts.columns if c in ["sport_name","activity_name","sport"]), None)
if sport_col:
    print(f"\nTop 10 sports:")
    print(workouts[sport_col].value_counts().head(10).to_string())

if "strain_score" in workouts.columns:
    s = workouts["strain_score"].dropna()
    print(f"\nStrain score: mean {s.mean():.2f}, median {s.median():.2f}, max {s.max():.2f}")
    print(f"  Light (0-9):     {((s >= 0) & (s < 10)).sum()}")
    print(f"  Moderate (10-13):{((s >= 10) & (s < 14)).sum()}")
    print(f"  High (14-17):    {((s >= 14) & (s < 18)).sum()}")
    print(f"  All-Out (18-21): {(s >= 18).sum()}")

In [ ]:
# Workout patterns figure
if "workout_end_dt" in workouts.columns and "strain_score" in workouts.columns:
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))

    # (a) Workout strain distribution
    ax = axes[0, 0]
    s = workouts["strain_score"].dropna()
    ax.hist(s, bins=25, edgecolor="white", alpha=0.85, color="#E15759")
    for boundary, label in [(10,"Moderate"), (14,"High"), (18,"All-Out")]:
        ax.axvline(boundary, color="black", linestyle=":", alpha=0.5)
    ax.set_title("(a) Per-Workout Strain Distribution")
    ax.set_xlabel("Strain Score (0-21)"); ax.set_ylabel("Workouts")

    # (b) Workouts by time of day
    ax = axes[0, 1]
    workouts["end_hour"] = workouts["workout_end_dt"].dt.hour
    hour_counts = workouts["end_hour"].value_counts().sort_index()
    ax.bar(hour_counts.index, hour_counts.values,
           color="#F28E2B", edgecolor="black", linewidth=0.5)
    ax.set_title("(b) Workout End Time (Hour of Day)")
    ax.set_xlabel("Hour (0-23)"); ax.set_ylabel("Workouts")
    ax.set_xticks(range(0, 24, 2))

    # (c) Workouts by day of week
    ax = axes[1, 0]
    workouts["end_dow"] = workouts["workout_end_dt"].dt.day_name()
    dow_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
    dow_counts = workouts["end_dow"].value_counts().reindex(dow_order).fillna(0)
    ax.bar(dow_order, dow_counts.values, color="#4E79A7", edgecolor="black", linewidth=0.5)
    ax.set_title("(c) Workouts by Day of Week")
    ax.set_xlabel(""); ax.set_ylabel("Workouts")
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

    # (d) Workout strain → next-day recovery
    ax = axes[1, 1]
    if "recovery_score" in timeline.columns:
        workouts["_end_date"] = workouts["workout_end_dt"].dt.date
        workouts["_next_day"] = workouts["_end_date"].apply(lambda d: d + timedelta(days=1))

        nextday_rec = timeline[["date","recovery_score"]].copy()
        nextday_rec.columns = ["_next_day","next_day_recovery"]
        merged = workouts.merge(nextday_rec, on="_next_day", how="left").dropna(
            subset=["strain_score","next_day_recovery"])

        if len(merged) > 5:
            ax.scatter(merged["strain_score"], merged["next_day_recovery"],
                       alpha=0.5, s=18, color="#E15759", edgecolor="white", linewidth=0.3)
            r = merged["strain_score"].corr(merged["next_day_recovery"])
            z = np.polyfit(merged["strain_score"], merged["next_day_recovery"], 1)
            xs = np.linspace(merged["strain_score"].min(), merged["strain_score"].max(), 50)
            ax.plot(xs, np.polyval(z, xs), color="#333", linewidth=2,
                    label=f"r = {r:.2f} (n={len(merged)})")
            ax.set_title("(d) Workout Strain → Next-Day Recovery")
            ax.set_xlabel("Workout Strain (0-21)"); ax.set_ylabel("Next-Day Recovery (%)")
            ax.legend()

    plt.suptitle("Workout & Strain Profile", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig_05_workout_profile.png", dpi=300, bbox_inches="tight")
    plt.show()

## 8. Correlation Matrix — What Predicts Recovery Already?Before adding music features, we should know which **existing Whoop features** correlate with recovery. This:- **Establishes a baseline feature set** for the ML models- **Identifies potential confounders** to control for when isolating the music effect- **Supports the "Originality" argument** — if sleep/strain already explain most variance, music must explain additional variance beyond themA feature with strong recovery correlation will be kept as a control variable in the RecoverWave model, so the music-effect estimate is net of these known influences.

In [ ]:
# Build a clean feature matrix for correlation
feature_cols = []
for candidate in [
    "recovery_score",
    "hrv_rmssd_milliseconds",
    "resting_heart_rate",
    "day_strain",
    "sleep_performance_percentage",
    "sleep_efficiency_percentage",
    "sleep_consistency_percentage",
    "respiratory_rate",
    "light_ratio", "deep_ratio", "rem_ratio",
    "kilojoules",
]:
    if candidate in timeline.columns:
        feature_cols.append(candidate)

corr_df = timeline[feature_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)
sns.heatmap(
    corr_df, mask=mask, annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.4, cbar_kws={"label": "Pearson r"},
    ax=ax,
)
ax.set_title("Feature Correlation Matrix (Whoop Only)", pad=12)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_06_correlation_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

# Which features correlate most strongly with recovery?
rec_corrs = corr_df["recovery_score"].drop("recovery_score").sort_values(key=abs, ascending=False)
print("Features most correlated with recovery_score:")
for feat, c in rec_corrs.items():
    print(f"  {feat:45s} {c:+.3f}")

## 9. Journal Entries — Lifestyle ConfoundersIf populated, journal entries reveal lifestyle variables that strongly influence recovery and must be controlled for:- **Alcohol** → reduces REM, lowers HRV, impairs recovery- **Caffeine (after 4pm)** → fragments sleep- **Illness / stress** → increases RHR, reduces HRV- **Late meal** → delays sleep onsetThese will become control variables in the model (or exclusion criteria for the test set if frequency allows).

In [ ]:
if len(journal) > 0:
    # Journal format: long-form (one row per question+answer per day)
    q_col = next((c for c in journal.columns if "question" in c.lower()), None)
    a_col = next((c for c in journal.columns if c.lower() in ["answer","response"]), None)

    if q_col:
        top_questions = journal[q_col].value_counts().head(15)
        print("Most frequent journal questions:")
        for q, n in top_questions.items():
            print(f"  [{n:>4}]  {q}")

        if a_col:
            # For boolean-style questions, show yes/no split
            print("\nAnswer patterns for top questions:")
            for q in top_questions.index[:8]:
                answers = journal[journal[q_col] == q][a_col].value_counts()
                print(f"\n  {q}")
                for a, n in answers.head(5).items():
                    print(f"     {str(a)[:60]:62s} {n:>4}")
else:
    print("No journal entries — either file was absent or you haven't used journal feature.")
    print("This is fine — Whoop recovery calculation does not depend on journal inputs.")

## 10. Data Readiness Check for RecoverWaveFinal check: do we have enough data to build the model once Spotify arrives? The alignment pipeline needs:- Workouts with **end timestamps** (for listening window start)- Recovery scores on the **day after** workouts (target)- Dense enough workout frequency to yield useful training rows (target: ≥50 complete rows)

In [ ]:
# How many workouts have a valid next-day recovery?
if "workout_end_dt" in workouts.columns and "recovery_score" in timeline.columns:
    rec_by_date = timeline[["date", "recovery_score"]].dropna().set_index("date")["recovery_score"].to_dict()

    workouts["_end_date"] = workouts["workout_end_dt"].dt.date
    workouts["_next_day"] = workouts["_end_date"].apply(lambda d: d + timedelta(days=1))
    workouts["next_day_recovery"] = workouts["_next_day"].map(rec_by_date)

    n_total    = len(workouts)
    n_with_end = workouts["workout_end_dt"].notna().sum()
    n_with_rec = workouts["next_day_recovery"].notna().sum()

    print("Data Readiness Summary")
    print("=" * 50)
    print(f"  Total workouts recorded:              {n_total:>5}")
    print(f"  Workouts with valid end time:         {n_with_end:>5}")
    print(f"  Workouts with next-day recovery:      {n_with_rec:>5}")
    print(f"  → Maximum aligned training rows:      {n_with_rec:>5}")
    print()
    if n_with_rec >= 100:
        print("  ✅ Excellent — plenty of data for LSTM + baseline comparison")
    elif n_with_rec >= 50:
        print("  ✅ Sufficient — LSTM viable with regularisation; baselines will be robust")
    elif n_with_rec >= 30:
        print("  ⚠️  Marginal — prefer baseline models; LSTM high overfitting risk")
    else:
        print("  ❌ Too few — consider lengthening listening window or supplementing with public datasets")

    print()
    print(f"Spotify data arrival will further reduce this by the fraction of")
    print(f"post-workout windows where you weren't listening (expect 50-80% retention).")

## 11. Export Processed Timeline for Downstream NotebooksSave the cleaned timeline and a summary report so subsequent notebooks (`02_spotify_eda.py`, `03_alignment.py`) can load them directly.

In [ ]:
# Save processed timeline
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

timeline_out = processed_dir / "whoop_daily_timeline.csv"
timeline.to_csv(timeline_out, index=False)
print(f"✓ Saved timeline:        {timeline_out}")

# Save workouts with end timestamps (needed for alignment)
if "workout_end_dt" in workouts.columns:
    workouts_out = processed_dir / "whoop_workouts_clean.csv"
    workouts.to_csv(workouts_out, index=False)
    print(f"✓ Saved workouts:        {workouts_out}")

# Save a human-readable summary for the dissertation appendix
summary_out = processed_dir / "whoop_eda_summary.txt"
with open(summary_out, "w") as f:
    f.write("RecoverWave — Whoop Personal Data Characterisation Summary\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Export date range: {date_min.date()} → {date_max.date()} ({span_days} days)\n")
    f.write(f"Days with recovery score: {timeline['recovery_score'].notna().sum()}\n")
    f.write(f"Total workouts: {len(workouts)}\n\n")
    if "recovery_score" in timeline.columns:
        f.write(f"Recovery score — mean {rec.mean():.1f}%, median {rec.median():.0f}%, std {rec.std():.1f}\n")
    if hrv_col:
        f.write(f"HRV RMSSD — mean {hrv.mean():.1f} ms, median {hrv.median():.0f} ms\n")
    if "strain_score" in workouts.columns:
        f.write(f"Workout strain — mean {workouts['strain_score'].mean():.2f}, max {workouts['strain_score'].max():.1f}\n")
    f.write(f"\nFigures saved to: {FIG_DIR}\n")
print(f"✓ Saved summary:         {summary_out}")
print(f"\n✓ Figures saved to:      {FIG_DIR}")

## 12. Key Takeaways for the DissertationFill in these values once you run the notebook — they form the quantitative skeleton of the *Design* section and the n=1 limitation discussion in the *Reliability & Validity* section.### Dataset Characterisation- **Temporal coverage:** `___` days, `___` with recovery scores- **Recovery distribution:** mean `___` %, std `___`, median `___`%- **HRV distribution:** median `___` ms (population reference: ~53 ms for ages 20-30)- **Sleep architecture:** deep `___`%, REM `___`%, light `___`% of total sleep- **Workout density:** `___` workouts, `___` per week on average- **Lag-1 recovery autocorrelation:** `___` (persistence baseline to beat)### Modelling Implications- **Maximum training rows** (workouts × next-day recovery): `___`- **Strongest existing predictor of recovery** (from correlation matrix): `___`- **Confounders to control for** in the music model: `___`### n=1 Discussion Material- Compare your personal distributions to population norms cited in *Section 5* (HRV) and *Section 6* (sleep)- The dissertation section on *Reliability & Validity* will argue that:  1. Personal baselines fall within published healthy adult ranges → model conclusions aren't driven by outlier physiology  2. Cross-validation of feature engineering on public datasets (PPG-DaLiA, EmoWear) supports generalisability of the *method* even if the *findings* are individual---### Next Steps (once Spotify data arrives)1. Run `02_spotify_eda.ipynb` to characterise listening history2. Run `03_alignment.ipynb` to build the workout → listening → recovery dataset3. Run `04_baseline_models.ipynb` to train RF + XGBoost4. Run `05_sequence_model.ipynb` to train the LSTM + attention model**References for dissertation Literature Survey:**- Nunan D et al. (2010). *A quantitative systematic review of normal values for short-term heart rate variability in healthy adults*. Pacing Clin Electrophysiol. [doi:10.1111/j.1540-8159.2010.02841.x](https://pubmed.ncbi.nlm.nih.gov/20718611/)- Hirshkowitz M et al. (2015). *National Sleep Foundation's sleep time duration recommendations*. Sleep Health. [doi:10.1016/j.sleh.2014.12.010](https://pubmed.ncbi.nlm.nih.gov/26551974/)- Whoop Developer Docs — [Cycle](https://developer.whoop.com/docs/developing/user-data/cycle/), [Sleep](https://developer.whoop.com/docs/developing/user-data/sleep/), [Workout](https://developer.whoop.com/docs/developing/user-data/workout/)